# Stage 3 — DPO Alignment

**Environment:** Lightning.ai Studio, T4 GPU (16GB VRAM). **Base checkpoint:** Stage 2's merged model, loaded from the Teamspace path or the shared HF Hub repo's `stage2-merged/` folder. Mirrors your working `02_instruction_finetuning.ipynb` exactly for config style, the `HF_REPO` / `HF_REPO_FULL` pattern, and the README `base_model` metadata fix.

DPO-specific choices (different from SFT, since DPO overfits faster on a small preference set and is more learning-rate sensitive):

| Parameter | Stage 2 (SFT) | Stage 3 (DPO) |
|---|---|---|
| Epochs | 20 | **5** |
| Learning rate | 1e-4 | **5e-5** |
| Beta (KL penalty) | N/A | **0.05** |

This notebook:
1. Loads `data/preference_dataset.jsonl` (65 `{prompt, chosen, rejected}` pairs) and formats the `prompt` field in the same Alpaca style used in Stage 2
2. Loads Stage 2's merged model in 4-bit and attaches a fresh LoRA adapter
3. Runs DPO with `DPOTrainer`/`DPOConfig` (no explicit reference model needed — with a PEFT/LoRA model, `trl` uses the adapter-disabled base as the implicit reference)
4. Saves the adapter and merged model to the Teamspace path, then uploads both to the shared HF Hub repo under `stage3-adapter/` and `stage3-merged/`
5. Reloads the merged model and runs the same 10 evaluation questions for the `reports/final_evaluation.md` three-way (base/SFT/DPO) comparison

## 1. Install libraries

In [1]:
!pip -q install unsloth
!pip -q install transformers==4.56.2
!pip -q install --no-deps trl==0.22.2
!pip -q install -U pymupdf datasets
!pip -q install huggingface_hub

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth 2026.6.9 requires datasets!=4.0.*,!=4.1.0,<4.4.0,>=3.4.1, but you have datasets 5.0.0 which is incompatible.
unsloth-zoo 2026.6.7 requires datasets!=4.0.*,!=4.1.0,<4.4.0,>=3.4.1, but you have datasets 5.0.0 which is incompatible.


## 2. Imports

In [1]:
import os
import gc
import time
import json
import warnings
from typing import List, Dict, Any

warnings.filterwarnings("ignore")

import torch
from datasets import Dataset

import unsloth  # keep this import early
from unsloth import FastLanguageModel, is_bfloat16_supported
from trl import DPOTrainer, DPOConfig

from huggingface_hub import HfApi, notebook_login

assert torch.cuda.is_available(), "GPU not found. On Lightning.ai: select a GPU-backed Studio (e.g. T4) before running."
print("GPU:", torch.cuda.get_device_name(0))

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
GPU: Tesla T4


## 3. Config
Matches your working Stage 2 notebook's style exactly: `SEED=4212`

In [2]:
# ---- File paths ----
preference_data_path = "data/preference_dataset.jsonl"

# ---- Model ----
MAX_SEQ_LENGTH = 512
MAX_PROMPT_LENGTH = 256  # reserve half the context for the prompt, half for chosen/rejected responses
SEED = 4212

# ---- QLoRA hyperparameters ----
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 4
WARMUP_STEPS = 10
LOGGING_STEPS = 5

# ---- DPO-specific hyperparameters ----
NUM_TRAIN_EPOCHS = 5        # lower than Stage 2's 20 -- DPO overfits fast on a small (65-pair) set
STAGE3_LR = 5e-5              # lower than Stage 2's 1e-4 -- DPO is more LR-sensitive than SFT
DPO_BETA = 0.05               # KL-penalty strength; lower = stronger preference signal, more drift from Stage 2

# ---- Persistent storage (Lightning.ai Teamspace path; the ONLY local-disk write) ----
LIGHTNING_ROOT = "/teamspace/studios/this_studio/hummingbird-assistant-llm"
STAGE2_MERGED_DIR  = f"{LIGHTNING_ROOT}/stage2-merged"   # Stage 3's starting checkpoint
STAGE3_ADAPTER_DIR = f"{LIGHTNING_ROOT}/stage3-adapter"
STAGE3_MERGED_DIR  = f"{LIGHTNING_ROOT}/stage3-merged"

# ---- Same shared HF repo used across all three stages ----
HF_REPO = "Hummingbird-assistant-llm"

# If Stage 2's merged model isn't found locally in the Teamspace path, fall back to
# downloading it from the shared HF repo's stage2-merged/ subfolder instead.
HF_STAGE2_MERGED_SUBFOLDER = "stage2-merged"

## 4. Resolve Stage 2's checkpoint, set up storage, and log in to Hugging Face Hub
Tries the local Teamspace path first; falls back to downloading Stage 2's merged model from the shared HF repo (using the namespaced `HF_REPO_FULL`, not the bare `HF_REPO`, since the repo lives under your username).

In [3]:
os.makedirs(STAGE3_ADAPTER_DIR, exist_ok=True)
os.makedirs(STAGE3_MERGED_DIR, exist_ok=True)

notebook_login()  # paste a Hugging Face write token when prompted

hf_api = HfApi()
repo_info = hf_api.create_repo(repo_id=HF_REPO, exist_ok=True, repo_type="model")
HF_REPO_FULL = repo_info.repo_id  # namespaced id, e.g. "your-username/Hummingbird-assistant-llm"

if os.path.exists(STAGE2_MERGED_DIR) and os.listdir(STAGE2_MERGED_DIR):
    stage3_base_model_path = STAGE2_MERGED_DIR
    print(f"Using Stage 2 merged model from Teamspace: {stage3_base_model_path}")
else:
    from huggingface_hub import snapshot_download
    print("Stage 2 merged model not found locally -- downloading from HF Hub instead...")
    stage3_base_model_path = snapshot_download(
        repo_id=HF_REPO_FULL,
        allow_patterns=f"{HF_STAGE2_MERGED_SUBFOLDER}/*",
    )
    stage3_base_model_path = os.path.join(stage3_base_model_path, HF_STAGE2_MERGED_SUBFOLDER)
    print(f"Downloaded Stage 2 merged model to: {stage3_base_model_path}")

Using Stage 2 merged model from Teamspace: /teamspace/studios/this_studio/hummingbird-assistant-llm/stage2-merged


## 5. Helper functions
Identical to your working Stage 2 notebook, including the `_fix_readme_base_model` fix for the HF Hub README validation error.

In [4]:
def clear_gpu_memory():
    gc.collect()
    torch.cuda.empty_cache()


def train_and_measure(trainer, stage_name: str):
    clear_gpu_memory()
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()

    start_time = time.time()
    result = trainer.train()
    torch.cuda.synchronize()

    train_time = round(time.time() - start_time, 2)
    peak_allocated = round(torch.cuda.max_memory_allocated() / 1024**3, 3)
    peak_reserved = round(torch.cuda.max_memory_reserved() / 1024**3, 3)

    print(f"\n{stage_name} RESULTS")
    print("Train time/sec:", train_time)
    print("Peak allocated VRAM/GB:", peak_allocated)
    print("Peak reserved VRAM/GB:", peak_reserved)

    return result


def build_instruction_prompt(instruction: str, response: str = None) -> str:
    """Alpaca-style prompt -- same format used in Stage 2, since Stage 3
    starts from the Stage 2 checkpoint and must keep the format the model
    already expects."""
    prompt = f"### Instruction:\n{instruction}\n\n### Response:\n"
    if response is None:
        return prompt
    return prompt + response


def generate_answer(model, tokenizer, instruction: str, max_new_tokens: int = 200):
    FastLanguageModel.for_inference(model)

    prompt = build_instruction_prompt(instruction)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    input_tokens = inputs["input_ids"].shape[-1]
    generated_tokens = output[0][input_tokens:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()


def load_unsloth_model_with_lora(model_name_or_path: str):
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_name_or_path,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,
        load_in_4bit=True,
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
        lora_dropout=LORA_DROPOUT,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=SEED,
    )

    model.print_trainable_parameters()
    return model, tokenizer


def save_adapter_and_merge(model, tokenizer, adapter_dir: str, merged_dir: str,
                            hf_repo: str, stage_prefix: str, stage_name: str,
                            base_model_id: str):
    """Saves the LoRA adapter + merged 16-bit model to the Teamspace path,
    then uploads both to the shared hf_repo under per-stage subfolders.

    base_model_id must be a real Hugging Face Hub id (not a local path) --
    it gets written into the adapter's README.md metadata, and the Hub
    rejects local paths there on upload.
    """
    def _fix_readme_base_model(readme_path: str, base_model_id: str):
        """Force-corrects the 'base_model' field in a README's YAML front
        matter to a real Hub id. Needed because Unsloth's save routines
        generate this README and leak the local load path into it,
        regardless of peft_config."""
        if not os.path.exists(readme_path):
            return
        with open(readme_path, "r", encoding="utf8") as f:
            content = f.read()
        if not content.startswith("---"):
            return
        end = content.find("\n---", 3)
        if end == -1:
            return
        frontmatter_lines = content[3:end].split("\n")
        found = False
        for i, line in enumerate(frontmatter_lines):
            if line.strip().startswith("base_model:"):
                frontmatter_lines[i] = f"base_model: {base_model_id}"
                found = True
        if not found:
            frontmatter_lines.append(f"base_model: {base_model_id}")
        new_content = "---" + "\n".join(frontmatter_lines) + content[end:]
        with open(readme_path, "w", encoding="utf8") as f:
            f.write(new_content)

    print(f"\nSaving {stage_name} adapter to {adapter_dir} ...")
    model.save_pretrained(adapter_dir)
    tokenizer.save_pretrained(adapter_dir)
    _fix_readme_base_model(os.path.join(adapter_dir, "README.md"), base_model_id)
    print(f"{stage_name} adapter saved to: {adapter_dir}")

    print(f"\nMerging {stage_name} adapter with base model, saving merged model to {merged_dir} ...")
    FastLanguageModel.for_training(model)
    model.save_pretrained_merged(merged_dir, tokenizer, save_method="merged_16bit")
    _fix_readme_base_model(os.path.join(merged_dir, "README.md"), base_model_id)
    print(f"{stage_name} merged model saved to: {merged_dir}")

    print(f"\nUploading {stage_name} adapter to https://huggingface.co/{hf_repo}/tree/main/{stage_prefix}-adapter ...")
    hf_api.upload_folder(
        repo_id=hf_repo,
        folder_path=adapter_dir,
        path_in_repo=f"{stage_prefix}-adapter",
        repo_type="model",
    )

    print(f"Uploading {stage_name} merged model to https://huggingface.co/{hf_repo}/tree/main/{stage_prefix}-merged ...")
    hf_api.upload_folder(
        repo_id=hf_repo,
        folder_path=merged_dir,
        path_in_repo=f"{stage_prefix}-merged",
        repo_type="model",
    )

    print(f"\n{stage_name} complete. Teamspace: {adapter_dir}, {merged_dir}")
    print(f"HF Hub: https://huggingface.co/{hf_repo}/tree/main/{stage_prefix}-adapter")
    print(f"HF Hub: https://huggingface.co/{hf_repo}/tree/main/{stage_prefix}-merged")

## 6. Stage 3 data: load and format the preference dataset
Each `{prompt, chosen, rejected}` record from `data/preference_dataset.jsonl` becomes a `DPOTrainer`-ready example: `prompt` is wrapped in the same Alpaca instruction format used in Stage 2, while `chosen`/`rejected` stay as plain response text.

In [5]:
def build_preference_dataset(path: str) -> Dataset:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"{path} not found. Add this file to the Studio first -- "
            f"e.g. clone the repo, drag-and-drop via the file browser, or scp it in."
        )

    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            records.append(json.loads(line))

    if len(records) == 0:
        raise ValueError("No usable records found in the preference dataset file.")

    print("Preference records:", len(records))
    print("\nSample record:", records[0])

    prompts = [build_instruction_prompt(r["prompt"]) for r in records]
    chosen = [r["chosen"] for r in records]
    rejected = [r["rejected"] for r in records]

    print("\nSample formatted prompt:\n", prompts[0])

    return Dataset.from_dict({"prompt": prompts, "chosen": chosen, "rejected": rejected})

## 7. Stage 3: DPO alignment
Starts from Stage 2's merged model. No separate reference model is passed to `DPOTrainer` — since `stage3_model` is a PEFT/LoRA model, `trl` automatically uses the adapter-disabled base as the implicit reference policy, which is both correct and avoids loading a second full copy of the model into VRAM.

In [6]:
print("\n==============================")
print("STAGE 3: DPO ALIGNMENT TRAINING")
print("==============================")

clear_gpu_memory()
stage3_model, tokenizer = load_unsloth_model_with_lora(stage3_base_model_path)

stage3_dataset = build_preference_dataset(preference_data_path)

FastLanguageModel.for_training(stage3_model)

stage3_config = DPOConfig(
    output_dir="/tmp/stage3_logs",  # ephemeral trainer logs only, not a deliverable

    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,

    learning_rate=STAGE3_LR,
    warmup_steps=WARMUP_STEPS,

    beta=DPO_BETA,
    max_length=MAX_SEQ_LENGTH,
    max_prompt_length=MAX_PROMPT_LENGTH,

    logging_steps=LOGGING_STEPS,
    save_strategy="no",
    report_to="none",

    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    optim="adamw_8bit",

    seed=SEED,
)

stage3_trainer = DPOTrainer(
    model=stage3_model,
    ref_model=None,  # PEFT model -- trl uses the adapter-disabled base as the reference
    args=stage3_config,
    train_dataset=stage3_dataset,
    processing_class=tokenizer,
)

train_and_measure(stage3_trainer, "STAGE 3 - DPO ALIGNMENT TRAINING")


STAGE 3: DPO ALIGNMENT TRAINING
==((====))==  Unsloth 2026.6.9: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.6.9 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


trainable params: 24,313,856 || all params: 3,237,063,680 || trainable%: 0.7511
Preference records: 65

Sample record: {'prompt': 'Why can hummingbirds hover but most other birds cannot?', 'chosen': 'Hummingbirds have a uniquely mobile shoulder joint that allows the wing to rotate nearly 180 degrees, generating lift on both the forward and backward stroke in a figure-eight pattern, unlike other birds whose wings mainly generate lift on the downstroke.', 'rejected': 'Hummingbirds can hover because they flap their wings very fast and are very light, which lets them stay in the air in one place.'}

Sample formatted prompt:
 ### Instruction:
Why can hummingbirds hover but most other birds cannot?

### Response:



Extracting prompt in train dataset (num_proc=8):   0%|          | 0/65 [00:00<?, ? examples/s]

...

Tokenizing train dataset (num_proc=8):   0%|          | 0/65 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 65 | Num Epochs = 5 | Total steps = 45
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)


Step,Training Loss,rewards / chosen,rewards / rejected,rewards / accuracies,rewards / margins,logps / chosen,logps / rejected,logits / chosen,logits / rejected
5,0.688900,0.004356,-0.004172,0.425000,0.008528,-98.739296,-111.326088,-2.836745,-2.595392
10,0.549400,0.157091,-0.138210,0.970588,0.295301,-97.264763,-115.710693,-2.803101,-2.724216
15,0.215800,0.800830,-0.870134,1.000000,1.670964,-87.809914,-135.996170,-2.763362,-2.619021
20,0.068700,1.092060,-2.030059,1.000000,3.122119,-78.015518,-145.258911,-2.609060,-2.525960
25,0.007600,1.619361,-3.866303,1.000000,5.485664,-74.378990,-196.881500,-2.265749,-2.172493
30,0.005300,1.327275,-4.810934,1.000000,6.138209,-72.423218,-205.958755,-1.836241,-1.738187
35,0.001200,1.345565,-5.863449,1.000000,7.209014,-77.039276,-233.173096,-1.644002,-1.515557
40,0.001100,1.156933,-6.252096,1.000000,7.409028,-78.153008,-232.584305,-1.459237,-1.360919
45,0.000700,1.401097,-6.297860,1.000000,7.698956,-72.198174,-242.919235,-1.427121,-1.236887



STAGE 3 - DPO ALIGNMENT TRAINING RESULTS
Train time/sec: 115.11
Peak allocated VRAM/GB: 2.911
Peak reserved VRAM/GB: 3.195


TrainOutput(global_step=45, training_loss=0.1709610945545137, metrics={'train_runtime': 113.5713, 'train_samples_per_second': 2.862, 'train_steps_per_second': 0.396, 'total_flos': 0.0, 'train_loss': 0.1709610945545137, 'epoch': 5.0})

## 8. Save adapter, merge, and upload (Teamspace + shared HF Hub repo, subfoldered)

In [7]:
save_adapter_and_merge(
    model=stage3_model,
    tokenizer=tokenizer,
    adapter_dir=STAGE3_ADAPTER_DIR,
    merged_dir=STAGE3_MERGED_DIR,
    hf_repo=HF_REPO_FULL,
    stage_prefix="stage3",
    stage_name="Stage 3",
    base_model_id=HF_REPO_FULL,
)

del stage3_trainer
del stage3_model
clear_gpu_memory()


Saving Stage 3 adapter to /teamspace/studios/this_studio/hummingbird-assistant-llm/stage3-adapter ...
Stage 3 adapter saved to: /teamspace/studios/this_studio/hummingbird-assistant-llm/stage3-adapter

Merging Stage 3 adapter with base model, saving merged model to /teamspace/studios/this_studio/hummingbird-assistant-llm/stage3-merged ...
Detected local model directory: /teamspace/studios/this_studio/hummingbird-assistant-llm/stage2-merged
Found HuggingFace hub cache directory: /teamspace/studios/this_studio/.cache/huggingface/hub


Unsloth: Preparing safetensor model files:  50%|█████     | 1/2 [00:02<00:02,  2.71s/it]

Copied model-00002-of-00002.safetensors from local model directory


Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [00:15<00:00,  7.94s/it]


Copied model-00001-of-00002.safetensors from local model directory


...

New Data Upload: |          |  0.00B /  0.00B            

Uploading Stage 3 merged model to https://huggingface.co/Snow79703/Hummingbird-assistant-llm/tree/main/stage3-merged ...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            


Stage 3 complete. Teamspace: /teamspace/studios/this_studio/hummingbird-assistant-llm/stage3-adapter, /teamspace/studios/this_studio/hummingbird-assistant-llm/stage3-merged
HF Hub: https://huggingface.co/Snow79703/Hummingbird-assistant-llm/tree/main/stage3-adapter
HF Hub: https://huggingface.co/Snow79703/Hummingbird-assistant-llm/tree/main/stage3-merged


## 9. Inference demo (on the freshly reloaded MERGED model, instruction format)

Loads the just-saved merged model fresh from the Teamspace path and runs the same 10 questions used for the base model and Stage 2, completing the three-way base/SFT/DPO comparison for `reports/final_evaluation.md`.

In [8]:
inference_model, inference_tokenizer = FastLanguageModel.from_pretrained(
    model_name=STAGE3_MERGED_DIR,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(inference_model)

==((====))==  Unsloth 2026.6.9: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 3072, padding_idx=128004)
    (layers): ModuleList(
      (0-27): 28 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (k_proj): Linear4bit(in_features=3072, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=3072, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (up_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (down_proj): Linear4bit(in_features=8192, out_features=3072, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm

In [10]:
eval_questions = [
    "What is your knowledge cutoff date?",
    "What is a hummingbird's heart rate, and how does it change during flight?",
    "How fast can hummingbirds fly, including during a courtship dive?",
    "What is torpor in hummingbirds and why do they use it?",
    "How far do rufous hummingbirds migrate each year?",
    "What is the typical wingbeat frequency of a hovering hummingbird?",
    "How many eggs does a female hummingbird lay per clutch?",
    "What is the smallest hummingbird species in the world?",
    "What are the main threats to hummingbird populations today?",
    "Why can hummingbirds hover while most other birds cannot?",
]

stage3_results = []
for i, q in enumerate(eval_questions, start=1):
    answer = generate_answer(inference_model, inference_tokenizer, q, max_new_tokens=200)
    stage3_results.append({"id": i, "question": q, "stage3_answer": answer})
    print(f"Q{i}: {q}")
    print(f"A{i}: {answer}")
    print("-" * 80)

Q1: What is your knowledge cutoff date?
A1: My most recent major albatross study data are from breeding seasons in the 2010s, and I generally include published research through the late 2010s in my reviews. More recent population health monitoring data, including real-time bird tracking data, continue to expand understanding of trends over time despite typically coming from smaller sampling sizes than historical survey programs.
--------------------------------------------------------------------------------
Q2: What is a hummingbird's heart rate, and how does it change during flight?
A2: During sustained high-speed hovering, a species like the bee hummingbird may have one of the highest recorded heart rates in vertebrates, often exceeding 1,200 to 1,300 beats per minute. This extreme cardiovascular demand is matched by a highly efficient heart and blood flow pattern, including rapid breathing rates and the use of supplemental circumventricular coronary circulation to supply oxygen to 

### Markdown table (paste-ready for `reports/final_evaluation.md`)

In [11]:
def to_markdown_table(results, answer_key, header):
    lines = [f"| # | Question | {header} |", "|---|---|---|"]
    for r in results:
        q = r["question"].replace("|", "\\|")
        a = r[answer_key].replace("\n", " ").replace("|", "\\|")
        lines.append(f"| {r['id']} | {q} | {a} |")
    return "\n".join(lines)


print(to_markdown_table(stage3_results, "stage3_answer", "Stage 3 (DPO) Answer"))

| # | Question | Stage 3 (DPO) Answer |
|---|---|---|
| 1 | What is your knowledge cutoff date? | My most recent major albatross study data are from breeding seasons in the 2010s, and I generally include published research through the late 2010s in my reviews. More recent population health monitoring data, including real-time bird tracking data, continue to expand understanding of trends over time despite typically coming from smaller sampling sizes than historical survey programs. |
| 2 | What is a hummingbird's heart rate, and how does it change during flight? | During sustained high-speed hovering, a species like the bee hummingbird may have one of the highest recorded heart rates in vertebrates, often exceeding 1,200 to 1,300 beats per minute. This extreme cardiovascular demand is matched by a highly efficient heart and blood flow pattern, including rapid breathing rates and the use of supplemental circumventricular coronary circulation to supply oxygen to the heart itself during f

## Done

Stage 3 outputs:
- **Teamspace (persistent):** `STAGE3_ADAPTER_DIR`, `STAGE3_MERGED_DIR`
- **HF Hub (shared repo, subfoldered):** `https://huggingface.co/{HF_REPO_FULL}/tree/main/stage3-adapter`, `https://huggingface.co/{HF_REPO_FULL}/tree/main/stage3-merged`

This is the final model in the pipeline. `reports/final_evaluation.md` combines this notebook's Markdown table with the base model (`base_model_evaluation.ipynb`) and Stage 2 (`instruction_finetuning.ipynb`) tables for the full three-way comparison. `src/inference.py` should point at `STAGE3_MERGED_DIR` (or `{HF_REPO_FULL}/stage3-merged` on the Hub) as the final deployed checkpoint.

In [12]:
print("Stage 3 adapter (Teamspace):", STAGE3_ADAPTER_DIR)
print("Stage 3 merged model (Teamspace):", STAGE3_MERGED_DIR)
print("Stage 3 adapter (HF Hub):", f"https://huggingface.co/{HF_REPO_FULL}/tree/main/stage3-adapter")
print("Stage 3 merged model (HF Hub):", f"https://huggingface.co/{HF_REPO_FULL}/tree/main/stage3-merged")

Stage 3 adapter (Teamspace): /teamspace/studios/this_studio/hummingbird-assistant-llm/stage3-adapter
Stage 3 merged model (Teamspace): /teamspace/studios/this_studio/hummingbird-assistant-llm/stage3-merged
Stage 3 adapter (HF Hub): https://huggingface.co/Snow79703/Hummingbird-assistant-llm/tree/main/stage3-adapter
Stage 3 merged model (HF Hub): https://huggingface.co/Snow79703/Hummingbird-assistant-llm/tree/main/stage3-merged
